# 3. Exploración de Gold

Propósito: Explorar los marts pre-agregados en gold.

In [1]:
# --- Bootstrap del entorno (Windows + VSCode) ---
# VSCode inyecta el .env del repo (con rutas Linux para JAVA_HOME/HADOOP_HOME)
# dentro del kernel; en Windows esas rutas no existen y Spark no arranca.
# Ademas los notebooks corren desde notebooks/, por lo que fijamos el cwd y el
# sys.path en la raiz del repo para que 'data/...' e 'import app' funcionen.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

if not os.path.isdir(os.environ.get("JAVA_HOME", "")):
    os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-21.0.11.10-hotspot"
_hadoop = os.environ.get("HADOOP_HOME", "")
if not (os.path.isabs(_hadoop) and os.path.isdir(_hadoop)):
    os.environ["HADOOP_HOME"] = str(ROOT / "lib" / "hadoop")

print("ROOT:", ROOT)
print("JAVA_HOME:", os.environ["JAVA_HOME"])

ROOT: d:\Universidad\CICLO_VII\BigData\Proyecto\EP-GDM-G6
JAVA_HOME: C:\Program Files\Java\jdk-21


In [2]:
from pyspark.sql import SparkSession, functions as F
from app.utils.spark import SparkClient

# Reutiliza la sesion activa si el kernel ya tiene una; si no, crea una con la
# config Windows-correcta de SparkClient (rutas nativas Hadoop, memoria driver).
# Asi re-ejecutar la celda no intenta crear un segundo SparkContext (SPARK-2243).
spark = SparkSession.getActiveSession() or SparkClient().get_session()
gold_path = "data/gold"  # cwd = raiz del repo (ver celda de bootstrap)
print("Spark", spark.version)

Spark 4.1.1


In [3]:
dim_cal = spark.read.parquet(f"{gold_path}/DIM_CALENDARIO.parquet")
dim_geo = spark.read.parquet(f"{gold_path}/DIM_GEOGRAFIA.parquet")
print(f"DIM_CALENDARIO: {dim_cal.count()} filas")
print(f"DIM_GEOGRAFIA: {dim_geo.count()} filas")

DIM_CALENDARIO: 312 filas
DIM_GEOGRAFIA: 1891 filas


In [4]:
mart_geo = spark.read.parquet(f"{gold_path}/MART_INGRESOS_GEOGRAFICO.parquet")
mart_geo.groupBy("Departamento").agg(F.sum("MontoRecaudado").alias("Total")).orderBy(F.col("Total").desc()).show(10)

+------------+--------------+
|Departamento|         Total|
+------------+--------------+
|        LIMA|30151868814.98|
|       CUSCO|18689458518.39|
|      ANCASH|17219813826.65|
|    AREQUIPA|14301276500.09|
|       PIURA| 9757479514.66|
| LA LIBERTAD| 8390106637.59|
|   CAJAMARCA| 6936795689.07|
|        PUNO| 5488495136.61|
|         ICA| 5485910920.72|
|       JUNIN| 5323239949.38|
+------------+--------------+
only showing top 10 rows


In [5]:
mart_ejec = spark.read.parquet(f"{gold_path}/MART_INGRESOS_EJECUTORA.parquet")
mart_ejec.orderBy(F.col("MontoRecaudado").desc()).select("Ejecutora", "Departamento", "MontoRecaudado").show(10)

+--------------------+------------+--------------+
|           Ejecutora|Departamento|MontoRecaudado|
+--------------------+------------+--------------+
|MUNICIPALIDAD MET...|        LIMA| 4320250573.04|
|MUNICIPALIDAD MET...|        LIMA| 2606435647.37|
|MUNICIPALIDAD MET...|        LIMA| 2093521182.05|
|MUNICIPALIDAD MET...|        LIMA| 1643532459.27|
|MUNICIPALIDAD DIS...|      ANCASH| 1206777424.54|
|MUNICIPALIDAD DIS...|      ANCASH| 1098773217.56|
|MUNICIPALIDAD DIS...|      ANCASH| 1073600408.87|
|MUNICIPALIDAD DIS...|      ANCASH|  861832296.65|
|MUNICIPALIDAD DIS...|       CUSCO|  782054540.12|
|MUNICIPALIDAD DIS...|       CUSCO|  580989837.60|
+--------------------+------------+--------------+
only showing top 10 rows


In [6]:
mart_predial = spark.read.parquet(f"{gold_path}/MART_PREDIAL.parquet")
print(f"MART_PREDIAL: {mart_predial.count()} filas")
mart_predial.select("Municipalidad", "FormularioTitulo").distinct().show(5)

MART_PREDIAL: 230942 filas
+--------------------+--------------------+
|       Municipalidad|    FormularioTitulo|
+--------------------+--------------------+
|MUNICIPALIDAD DIS...|2D. MACROPROCESO ...|
|MUNICIPALIDAD DIS...|2C. MACROPROCESO ...|
|MUNICIPALIDAD DIS...|2C. MACROPROCESO ...|
|MUNICIPALIDAD PRO...|2C. MACROPROCESO ...|
|MUNICIPALIDAD PRO...|2B. MACROPROCESO ...|
+--------------------+--------------------+
only showing top 5 rows


In [7]:
mart_renamu = spark.read.parquet(f"{gold_path}/MART_RENAMU.parquet")
print(f"MART_RENAMU: {mart_renamu.count()} filas")
# EsAfirmativo es BOOLEAN; lo casteamos a int para promediar (avg = proporcion de True = cobertura).
mart_renamu.select("Departamento", "Descripcion", "EsAfirmativo").groupBy("Departamento").agg(
    F.avg(F.col("EsAfirmativo").cast("int")).alias("Cobertura")
).orderBy(F.col("Cobertura").desc()).show(10)

MART_RENAMU: 12770291 filas
+--------------------+-------------------+
|        Departamento|          Cobertura|
+--------------------+-------------------+
|PROVINCIA CONSTIT...|0.21931322329728703|
|             UCAYALI| 0.1817651680206047|
|               PIURA| 0.1808129306137274|
|          LAMBAYEQUE|0.17946925747303186|
|               PASCO|0.17838734461896985|
|              TUMBES|0.17669492968297748|
|               CUSCO|0.17519405833354823|
|                 ICA|0.17173682530858478|
|       MADRE DE DIOS|0.16798798073698473|
|                LIMA|0.16405765286266671|
+--------------------+-------------------+
only showing top 10 rows
